# Typed AAM, R/P alignment, and TS processing

This notebook uses the public typed API from beginning to end. AAM owns graph search and exact fragment symmetry. Analytical compilation, chirality/RMSD selection, and TS scoring consume the returned objects; no dictionary pipeline or fallback remapping is involved.

In [1]:
import numpy as np
import rxn_core
from rxn_core import (
    AAMProblem, AAMSearchConfig, MolecularEndpoint,
    TransitionStateTarget, VibrationalModes,
    search_aam, compile_mapping_families, select_rp_mappings,
    analyze_transition_state, align_reaction,
)
print('rxn_core', rxn_core.__version__)

rxn_core 0.2.0


## 1. Construct immutable molecular endpoints

Each endpoint contains elements, Cartesian coordinates, and a complete symmetric WBO matrix. The example changes two bonds and uses distinct elements so its expected mapping is easy to audit.

In [2]:
elements = ('C', 'N', 'O', 'H')
xyz = np.array([[0.,0.,0.], [1.,0.,0.], [0.,1.,0.], [1.,1.,0.2]])

def wbo_matrix(edges):
    matrix = np.zeros((len(elements), len(elements)))
    for left, right, value in edges:
        matrix[left, right] = matrix[right, left] = value
    return matrix

reactant = MolecularEndpoint(
    elements, xyz, wbo_matrix([(0,1,1.), (2,3,1.)]), label='R')
product = MolecularEndpoint(
    elements, xyz + 0.1, wbo_matrix([(0,2,1.), (1,3,1.)]), label='P')
problem = AAMProblem(reactant, product, name='tutorial_reaction')
print(problem.name, problem.atom_count, 'atoms')

tutorial_reaction 4 atoms


## 2. Run AAM and inspect its complete result

`workers` controls cut-sweep processes. `seed_count` is the number of deterministic seed orders per cut, while `branch_limit` bounds each growth subtree. Exact completed fragment generators are finalized inside AAM and recorded in its metrics.

In [3]:
config = AAMSearchConfig(seed_count=1, branch_limit=100, symmetry_repair=False)
aam = search_aam(problem, config, workers=1)
print('mechanisms:', len(aam.mechanisms))
print('completed branches:', sum(len(m.branches) for m in aam.mechanisms))
print('fragment-group calculations:', aam.metrics.completed_group_calculations)
branch = aam.mechanisms[0].branches[0]
print('fragment atoms:', [f.r_atoms for f in branch.hierarchy.fragments])
print('exact groups present:', branch.hierarchy.has_complete_exact_target_groups)

mechanisms: 1
completed branches: 3
fragment-group calculations: 4
fragment atoms: [(2,), (3,), (1,), (0,)]
exact groups present: True


## 3. Compile analytical families, then select R/P mapping

Family compilation deduplicates exact mapping cosets analytically. R/P selection applies chirality constraints to the generator representation and performs the exact covariance/RMSD search. It does **not** enumerate all molecular bijections. `workers` and `post_workers` are separate because AAM and family compilation scale differently.

In [4]:
families = compile_mapping_families(aam, workers=1, minimum_events_only=True)
rp = select_rp_mappings(families)
for mechanism in rp.mechanisms:
    print('mapping:', mechanism.mapping.as_dict())
    print('broken:', mechanism.broken_bonds, 'formed:', mechanism.formed_bonds)
    print('fixed-mapping RMSD:', round(mechanism.fixed_mapping_rmsd, 6))
    print('chirality violations:', mechanism.chirality['selected_index_chirality_violation_count'])

mapping: {0: 0, 1: 1, 2: 2, 3: 3}
broken: ((0, 1), (2, 3)) formed: ((0, 2), (1, 3))
fixed-mapping RMSD: 0.0
chirality violations: 0


In [5]:
# Convenience composition for ordinary R/P use:
rp_one_call = align_reaction(
    problem, search_config=config, workers=1, post_workers=1)
print(rp_one_call.mechanisms[0].mapping == rp.mechanisms[0].mapping)

True


### Choosing CPU counts

For a single case on a 48-core node, start with `workers=48`. `post_workers=8` is normally sufficient for small family sets; use up to 48 when AAM returns at least 128 unique family payloads. For many independent cases, allocate fewer workers per case and schedule cases concurrently. Worker count changes scheduling only, not chemistry or precision.

## 4. Mechanism-local TS processing

TS processing performs independent R→TS and P→TS partial AAM searches. Product assignments are pulled into R indexing, exact correlated core tuples are merged, and imaginary modes are scored. These are core assignments—not full-bijection enumeration.

In [6]:
target_molecule = MolecularEndpoint(
    elements, xyz + 0.05,
    wbo_matrix([(0,1,.5), (2,3,.5), (0,2,.5), (1,3,.5)]),
    label='TS')
modes = np.zeros((1, 4, 3))
modes[0, :, 0] = (1., -1., -1., 1.)
target = TransitionStateTarget(
    target_molecule, VibrationalModes(np.array([-500.]), modes))
ts = analyze_transition_state(rp, target, search_config=config)
selected = ts.mechanisms[0].selected
print('core assignment:', selected.assignment.as_dict())
print('endpoint support:', sorted(selected.sources))
print('mode/frequency:', selected.mode_index, selected.frequency)
print('score:', round(selected.score, 8))

core assignment: {0: 0, 1: 1, 2: 2, 3: 3}
endpoint support: ['product', 'reactant']
mode/frequency: 0 -500.0
score: 0.00344951


## 5. CLI and artifacts

The typed CLI accepts NPZ endpoints containing `elements`, `coordinates`, and `wbo`, or existing xTB cache directories. It writes `rp.json`, per-mechanism `R.xyz` and `P_aligned.xyz`, plus a self-contained `view.html`.

```bash
rxn-core --reactant-npz R.npz --product-npz P.npz \
  --workers 48 --post-workers 8 --output alignment

# Existing xTB caches are also accepted:
rxn-core --reactant-cache cache/R --product-cache cache/P \
  --workers 48 --output alignment
```